# Silver — Tempos dos chamados
Uma linha por chamado; todas as durações em segundos. Timestamps vêm de
bronze.chamados_hora e durações informadas de bronze.chamados.
O cliente fica em silver.chamados. A auditoria registra divergências da origem.

Período permitido: 01/01/2025 a 31/07/2026, em America/Sao_Paulo.
Datas fora do período ficam nulas com flags abertura/inicio/finalizacao_fora_periodo.
A Bronze preserva os valores originais. Início e fim válidos são preservados
mesmo quando a abertura é inválida; não se inventa uma data substituta.


In [ ]:
import json
from config import BRONZE_SCHEMA, SILVER_SCHEMA
from tempos import construir_tempos

spark.conf.set('spark.sql.session.timeZone', 'America/Sao_Paulo')
chamados = spark.table(f'{SILVER_SCHEMA}.chamados')
tempos, auditoria = construir_tempos(
    chamados, spark.table(f'{BRONZE_SCHEMA}.chamados'),
    spark.table(f'{BRONZE_SCHEMA}.chamados_hora'),
)
print(json.dumps(auditoria, ensure_ascii=False, sort_keys=True))
# A construção materializa as validações antes da publicação.
(tempos.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
 .saveAsTable(f'{SILVER_SCHEMA}.chamados_hora'))
